<a href="https://colab.research.google.com/github/ImSayvi/Sztuczna-Inteligencja/blob/main/lab7_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
!pip install -q transformers accelerate sentencepiece tokenizers faiss-cpu \
             sentence-transformers pymupdf tqdm ipywidgets huggingface_hub
print('Biblioteki zainstalowane!')

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU dostępne: {torch.cuda.get_device_name(0)}')
    print(f'   CUDA: {torch.version.cuda}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Brak GPU – praca na CPU (wolniej). Włącz GPU ')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUrządzenie: {DEVICE}')

In [ ]:
import os

os.makedirs('knowledge', exist_ok=True)
os.makedirs('vec_db', exist_ok=True)

# Pobieramy przykładowe artykuły naukowe (arxiv – open access)
pdfs_to_download = {
    'exoplanets.pdf': 'https://arxiv.org/pdf/astro-ph/9808116',
    'machine_learning_astro.pdf': 'https://arxiv.org/pdf/1904.07248',
}

import urllib.request

for filename, url in pdfs_to_download.items():
    path = f'knowledge/{filename}'
    if not os.path.exists(path):
        print(f'⬇️  Pobieranie {filename}...')
        try:
            urllib.request.urlretrieve(url, path)
            print(f'   ✅ Zapisano: {path}')
        except Exception as e:
            print(f'   ❌ Błąd: {e}')
    else:
        print(f'   ℹ️  {filename} już istnieje')

# Sprawdzamy co mamy
pdf_files = [f for f in os.listdir('knowledge') if f.endswith('.pdf')]
print(f'\n📚 Pliki PDF w knowledge/: {pdf_files}')

In [ ]:
# Sprawdź jakie masz pliki PDF
pdf_files = [f for f in os.listdir('knowledge') if f.endswith('.pdf')]
if pdf_files:
    print('Znalezione pliki PDF:')
    for f in pdf_files:
        size = os.path.getsize(f'knowledge/{f}') / 1024
        print(f'   • {f}  ({size:.0f} KB)')
else:
    print('Brak plików PDF w katalogu knowledge/')

In [ ]:
import fitz  # PyMuPDF
from tqdm import tqdm

def extract_chunks_from_pdfs(knowledge_dir='knowledge', chunk_size=500, overlap=50):
    """
    Wczytuje wszystkie PDF z katalogu i dzieli tekst na fragmenty (chunki).

    chunk_size  – ile znaków ma mieć każdy fragment
    overlap     – ile znaków nakłada się między sąsiednimi fragmentami
                  (dzięki temu nie gubimy kontekstu na granicach)
    """
    chunks = []  # lista słowników: {'text': ..., 'source': ..., 'page': ...}

    pdf_files = [f for f in os.listdir(knowledge_dir) if f.endswith('.pdf')]

    if not pdf_files:
        print('Brak plików PDF! Wgraj je do katalogu knowledge/')
        return chunks

    for pdf_name in tqdm(pdf_files, desc='📄 Ekstrakcja PDF'):
        pdf_path = os.path.join(knowledge_dir, pdf_name)
        doc = fitz.open(pdf_path)

        for page_num, page in enumerate(doc):
            text = page.get_text()
            text = text.strip()

            if not text or len(text) < 50:  # pomijamy puste/krótkie strony
                continue

            # Dzielimy stronę na mniejsze chunki jeśli jest długa
            if len(text) <= chunk_size:
                chunks.append({
                    'text': text,
                    'source': pdf_name,
                    'page': page_num + 1
                })
            else:
                # Sliding window – przesuwamy się o (chunk_size - overlap)
                step = chunk_size - overlap
                for i in range(0, len(text) - overlap, step):
                    chunk_text = text[i:i + chunk_size]
                    if len(chunk_text) > 50:
                        chunks.append({
                            'text': chunk_text,
                            'source': pdf_name,
                            'page': page_num + 1
                        })
        doc.close()

    return chunks


chunks = extract_chunks_from_pdfs()

print(f'\nWyekstrahowano {len(chunks)} fragmentów tekstu')
if chunks:
    print(f'\nPrzykładowy fragment:')
    print(f'   Źródło: {chunks[0]["source"]} – strona {chunks[0]["page"]}')
    print(f'   Tekst: {chunks[0]["text"][:200]}...')

In [ ]:
from sentence_transformers import SentenceTransformer

# Model do tworzenia embeddingów
# Dla tekstów polskich można użyć: 'sdadas/mmlw-retrieval-roberta-large'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

print(f'⬇️  Pobieranie modelu embeddingów: {EMBEDDING_MODEL}')
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print(f'✅ Model embeddingów gotowy!')

# Test – sprawdzamy wymiar embeddingu
test_emb = embedder.encode(['test'])
print(f'   Wymiar wektora: {test_emb.shape[1]}')

In [ ]:
import faiss
import numpy as np
import pickle

def build_vector_db(chunks, embedder, db_dir='vec_db'):
    """
    Tworzy bazę wektorową FAISS z listy chunków.
    Zapisuje indeks i metadane na dysk.
    """
    print(f'Tworzenie embeddingów dla {len(chunks)} fragmentów...')

    # Wyciągamy same teksty
    texts = [c['text'] for c in chunks]

    # Tworzymy embeddingi dla wszystkich chunków naraz (batch=32)
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Normalizacja – wymagana dla podobieństwa cosinusowego
    faiss.normalize_L2(embeddings)

    # Tworzymy indeks FAISS (Inner Product = kosinusowe po normalizacji)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    # Zapisujemy na dysk
    faiss.write_index(index, f'{db_dir}/faiss.index')
    with open(f'{db_dir}/chunks.pkl', 'wb') as f:
        pickle.dump(chunks, f)

    print(f'\n Baza wektorowa zapisana w {db_dir}/')
    print(f'   Liczba wektorów: {index.ntotal}')
    print(f'   Wymiar: {dim}')

    return index, chunks


faiss_index, all_chunks = build_vector_db(chunks, embedder)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

LLM_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print(f'⬇️  Pobieranie modelu LLM: {LLM_MODEL}')
print('   (może potrwać 3-5 minut za pierwszym razem)')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto',      # automatycznie rozdziela na GPU/CPU
    trust_remote_code=True
)

# Pipeline upraszcza generowanie tekstu
llm_pipeline = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

print('\n✅ Model LLM gotowy!')

In [ ]:
def retrieve(question, index, chunks, embedder, top_k=3):
    """
    Wyszukuje TOP-K najbardziej trafnych fragmentów dla danego pytania.
    """
    # Zamieniamy pytanie na wektor
    q_emb = embedder.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    # Szukamy w FAISS
    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:  # -1 = nie znaleziono
            results.append({
                'text': chunks[idx]['text'],
                'source': chunks[idx]['source'],
                'page': chunks[idx]['page'],
                'score': float(score)
            })
    return results


def build_prompt(question, retrieved_chunks):
    """
    Buduje prompt dla modelu LLM.
    KLUCZOWE: instruujemy model, by odpowiadał TYLKO na podstawie kontekstu.
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[{i}] Source: {chunk['source']}, page {chunk['page']}\n{chunk['text']}"
        )
    context = '\n\n'.join(context_parts)

    # Prompt w formacie chat (system + user)
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a helpful assistant that answers questions ONLY based on the provided context. '
                'If the context does not contain enough information to answer the question, '
                'say so clearly. Do NOT use any outside knowledge. '
                'Always cite the source number [1], [2], etc. when you use information from it.'
            )
        },
        {
            'role': 'user',
            'content': f'Context:\n{context}\n\nQuestion: {question}'
        }
    ]
    return messages


def ask_rag(question, index=faiss_index, chunks=all_chunks,
            embedder=embedder, llm=llm_pipeline, top_k=3, verbose=True):
    """
    Główna funkcja systemu RAG.
    """
    print(f'\nPytanie: {question}')
    print('-' * 60)

    # 1. Wyszukiwanie
    retrieved = retrieve(question, index, chunks, embedder, top_k)

    if verbose:
        print(f'🔍 Znaleziono {len(retrieved)} fragmentów:')
        for r in retrieved:
            print(f'   [{r["source"]} – str. {r["page"]}]  score={r["score"]:.3f}')
        print()

    # 2. Budowanie promptu
    messages = build_prompt(question, retrieved)

    # 3. Generowanie odpowiedzi
    print('🤖 Generowanie odpowiedzi...')
    output = llm(messages)
    answer = output[0]['generated_text'].strip()

    print(f'\n💬 Odpowiedź:\n{answer}')

    print('\n📚 Źródła:')
    for i, r in enumerate(retrieved, 1):
        print(f'   [{i}] {r["source"]} – strona {r["page"]}')
        print(f'       {r["text"][:120]}...')

    return answer, retrieved


print('System RAG gotowy! Możesz teraz zadawać pytania.')

In [ ]:
# Pytanie 1 – powinno być w dokumentach
answer, sources = ask_rag(
    'What are exoplanets and how are they detected?'
)

In [ ]:
# Pytanie 2 – powinno NIE być w dokumentach (test granic systemu)
# Dobrze skonfigurowany RAG powie, że nie ma takiej informacji
answer2, sources2 = ask_rag(
    'What is the recipe for Polish bigos?'
)

model mi halucynuje, bo korzysta z wytrenowanej wiedzy; problem małych modeli; ale jakby nie patrzeć odpowiedź jest prawie ok :)

In [ ]:
# Pytanie 3
moje_pytanie = 'How is machine learning used in astronomy?'

answer3, sources3 = ask_rag(moje_pytanie)